# 강원생활도우미(SDD) 3.0 — 막국수 맛집 추천 앱

**전체 기능:** 장소정보·추천정보 두 시트를 받아 ① place_id로 결합하고(join_data) → ② 조건에 맞는 맛집을 검색한다(search_places).

---
## 1. 프로젝트 명세 (제공)

| 항목 | 작성 내용 |
|------|-----------|
| AI의 역할 | Python·pandas로 강원생활도우미 앱을 함께 만드는 친절한 코딩 파트너 |
| 개발 목표 | 막국수 맛집 추천기 — 두 시트를 결합해 조건에 맞는 맛집을 검색 |
| 사용할 도구 | Python, pandas, Colab |
| 데이터 구조 | 장소정보(place_id, 이름, 지역, 평점, 주소 등), 추천정보(추천ID, place_id, 추천목적, 추천상황, 추천대상, 예산, 매운맛선택, 단체석여부 등) |
| 구현할 기능 | ① join_data: 두 시트를 place_id로 결합 ② search_places: 7개 조건(지역/추천목적/추천상황/추천대상/예산/매운맛선택/단체석여부)으로 검색 |
| 코드 수준 | 고등학생이 이해할 수 있는 수준 |
| 출력 형식 | 함수 단위 코드 + 짧은 설명 |

---
## 0. 데이터 불러오기

Colab에 `gangwon_makguksu_data.xlsx` 파일을 업로드한 뒤 아래 셀을 실행하세요.

In [ ]:
from google.colab import files
uploaded = files.upload()  # gangwon_makguksu_data.xlsx 선택

In [ ]:
import pandas as pd

place_df = pd.read_excel("gangwon_makguksu_data.xlsx", sheet_name="장소정보")
recommend_df = pd.read_excel("gangwon_makguksu_data.xlsx", sheet_name="추천정보")

print("장소정보:", place_df.shape)
print("추천정보:", recommend_df.shape)
place_df.head()

---
## 2. 분해 + 인터페이스 명세

**분해:** `장소정보, 추천정보` → **join_data** → `결합된 표` → **search_places** → `검색 결과`

### 인터페이스 명세
- join_data 의 출력: 추천정보의 모든 열 + 장소정보의 모든 열이 합쳐진 DataFrame (place_id 기준 결합)
- search_places 의 입력: 위 출력을 그대로 받는다
- 일치해야 할 열 이름: **'지역', '추천목적', '추천상황', '추천대상', '예산', '매운맛선택', '단체석여부'**

---
## 3. 사이클 1 — join_data

### 3-1. 기능 명세
- 무엇을: 장소정보와 추천정보를 place_id 기준으로 결합한다
- 입력: place_df(장소정보), recommend_df(추천정보) 두 개의 DataFrame
- 출력: 두 표가 합쳐진 하나의 DataFrame (인터페이스 명세와 일치)
- 제약: 추천정보 기준으로 결합한다 (한 장소에 추천이 여러 개 있을 수 있으므로). place_id가 없는 추천정보는 결과에서 제외하지 않고 그대로 남긴다(왼쪽 결합)
- 완료 조건: 결합 결과의 행 수 = 추천정보의 행 수, place_id가 양쪽에 모두 존재하면 이름·지역 등 장소정보 열이 정상적으로 채워져야 함

### 3-2. AI 프롬프트
```
다음 명세대로 pandas 함수를 만들어줘.
- 무엇을: 장소정보와 추천정보를 place_id 기준으로 결합한다
- 입력: place_df(장소정보), recommend_df(추천정보)
- 출력: 두 표가 합쳐진 DataFrame (함수: join_data(place_df, recommend_df))
- 제약: 추천정보 기준 왼쪽 결합(how='left'), place_id로 merge
- 완료 조건: 결합 결과 행 수 = 추천정보 행 수
고등학생이 이해할 수 있는 코드로, 설명은 짧게.
```

In [ ]:
# 3-3. 실행 코드
def join_data(place_df, recommend_df):
    # 추천정보를 기준으로 장소정보를 place_id로 결합한다
    joined = pd.merge(recommend_df, place_df, on='place_id', how='left')
    return joined

joined_df = join_data(place_df, recommend_df)
joined_df.head()

In [ ]:
# 3-4. 단위 검증
print('=== join_data 단위 검증 ===')

# ① 결합 결과의 행 수는 추천정보의 행 수와 같아야 한다
print('행 수 일치 :', len(joined_df) == len(recommend_df))

# ② 장소정보 열(이름, 지역)이 정상적으로 채워졌는가
print('이름 결측 없음 :', joined_df['이름'].isna().sum() == 0)
print('지역 결측 없음 :', joined_df['지역'].isna().sum() == 0)

# ③ 첫 행이 올바른 장소와 연결됐는가 (place_id=1 → 강릉 첫 번째 장소)
first_place_id = recommend_df.iloc[0]['place_id']
expected_name = place_df[place_df['place_id'] == first_place_id]['이름'].values[0]
print('첫 행 매칭 정확 :', joined_df.iloc[0]['이름'] == expected_name)

---
## 4. 사이클 2 — search_places

### 4-1. 기능 명세
- 무엇을: 결합된 표에서 사용자가 고른 조건에 맞는 맛집만 골라낸다
- 입력: join_data의 출력(DataFrame), 검색 조건들(지역, 추천목적, 추천상황, 추천대상, 예산상한, 매운맛선택, 단체석여부) — 각 조건은 '전체' 또는 비워두면 해당 조건은 건너뛴다
- 출력: 조건에 맞는 행만 남은 DataFrame
- 제약: 예산은 입력한 금액 **이하**인 행만 남긴다 (df['예산'] <= 예산상한). 나머지 조건은 정확히 일치하는 값만 남긴다
- 완료 조건: 조건을 하나도 안 주면 원본과 같은 결과가 나오고, 조건에 맞는 게 하나도 없으면 빈 DataFrame을 에러 없이 반환한다

### 4-2. AI 프롬프트
```
다음 명세대로 pandas 함수를 만들어줘.
- 무엇을: 결합된 표에서 조건에 맞는 맛집만 골라낸다
- 입력: joined_df, 지역, 추천목적, 추천상황, 추천대상, 예산상한, 매운맛선택, 단체석여부 (각 조건은 기본값 '전체')
- 출력: 조건에 맞는 행만 남은 DataFrame (함수: search_places(joined_df, ...))
- 제약: 예산은 예산상한 이하인 행만, 나머지는 정확히 일치하는 값만 필터링
- 완료 조건: 조건이 모두 '전체'면 원본 그대로 반환, 결과 없으면 빈 DataFrame 반환
고등학생이 이해할 수 있는 코드로, 설명은 짧게.
```

In [ ]:
# 4-3. 실행 코드
def search_places(joined_df,
                   지역='전체', 추천목적='전체', 추천상황='전체',
                   추천대상='전체', 예산상한=None,
                   매운맛선택='전체', 단체석여부='전체'):
    result = joined_df.copy()

    if 지역 != '전체':
        result = result[result['지역'] == 지역]
    if 추천목적 != '전체':
        result = result[result['추천목적'] == 추천목적]
    if 추천상황 != '전체':
        result = result[result['추천상황'] == 추천상황]
    if 추천대상 != '전체':
        result = result[result['추천대상'] == 추천대상]
    if 예산상한 is not None:
        result = result[result['예산'] <= 예산상한]
    if 매운맛선택 != '전체':
        result = result[result['매운맛선택'] == 매운맛선택]
    if 단체석여부 != '전체':
        result = result[result['단체석여부'] == 단체석여부]

    return result

# 예시: 강릉 + 여행 상황 + 예산 1만원 이하
search_places(joined_df, 지역='강릉', 추천상황='여행', 예산상한=10000)

In [ ]:
# 4-4. 단위 검증
print('=== search_places 단위 검증 ===')

# ④ 조건을 아무것도 안 주면 원본과 행 수가 같아야 한다
no_filter = search_places(joined_df)
print('조건 없음 = 원본 :', len(no_filter) == len(joined_df))

# ⑤ 지역 필터가 정확히 동작하는가 (결과가 전부 '강릉'인가)
gangneung_only = search_places(joined_df, 지역='강릉')
print('지역 필터 정확 :', (gangneung_only['지역'] == '강릉').all())

# ⑥ 예산 필터가 정확히 동작하는가 (결과가 전부 예산 이하인가)
cheap = search_places(joined_df, 예산상한=9000)
print('예산 필터 정확 :', (cheap['예산'] <= 9000).all())

# ⑦ 결과가 없을 때 에러 없이 빈 DataFrame을 반환하는가
empty = search_places(joined_df, 지역='서울')
print('빈 결과 처리 :', len(empty) == 0)

---
## 5. 통합 검증

In [ ]:
print('=== 통합 검증 ===')

# ⑧ join_data→search_places까지 이어 돌리기
joined = join_data(place_df, recommend_df)
final = search_places(joined, 지역='속초', 추천상황='더운날', 예산상한=10000, 단체석여부='O')
print('최종 검색 결과:')
final[['이름','지역','추천상황','대표메뉴가격','예산','단체석여부']]

In [ ]:
# ⑨ 인터페이스 열이 모두 존재하는가
필요열 = ['지역','추천목적','추천상황','추천대상','예산','매운맛선택','단체석여부']
print('인터페이스 열 모두 존재 :', all(c in joined.columns for c in 필요열))